# 🏢 SIMULACIÓN 3D INTERACTIVA: SISTEMA DE TRANSPORTE DE INFORMACIÓN EN DATA CENTER
### **IESTP HERMANOS CÁRCAMO – PAITA**
**Programa Académico:** Arquitectura de Plataformas y Servicios de TI  
**Módulo Formativo:** Módulo 3: Diseño de Arquitectura de Infraestructura y Plataforma de TI  
**Unidad Didáctica:** Diseño de Data Center – Sesión de Aprendizaje N° 05  
**Docente:** Ing. Javier Eduardo Jaramillo Atoche  
**Estudiante:** Gerson Misael Pintado Huaman  
**Año Lectivo:** Paita – Perú, 2026

---

## 🎯 OBJETIVO DE LA SIMULACIÓN
Desarrollar mediante programación en Python y visualización 3D interactiva la arquitectura física y lógica del sistema de transporte de información para un Centro de Datos empresarial (**Mar del Norte Logistics S.A.C. – Paita**).
La simulación modela:
1. **Zonificación Espacial (ANSI/TIA-942):** Cuarto de Entrada (ER), Distribución Principal (MDA), Distribución Horizontal (HDA) y Distribución de Equipos (EDA).
2. **Topología Spine-Leaf no bloqueante:** Switches Core Spine (100G) interconectados en malla completa con switches Leaf ToR (10G/40G).
3. **Medios de Transmisión Físicos:**
   - Troncales de **Fibra Óptica OM4 MTP/MPO** (100 Gbps) para el backbone.
   - Enlace externo de **Fibra Óptica Monomodo OS2** (WAN / Acometida portuaria).
   - Enlaces horizontales de **Cobre Categoría 6A apantallado** (10 Gbps) para parcheo de servidores en rack.
4. **Infraestructura de Canalización y Climatización:** Bandejas portacables tipo canastilla aérea, gabinetes 42U y confinamiento de pasillo frío / caliente (Cold Aisle Containment).


In [ ]:
# Instalación de dependencias para Google Colab o ejecución local
!pip install -q plotly numpy pandas

import numpy as np
import plotly.graph_objects as go
import json
print("✅ Entorno listo para la simulación 3D del Data Center")


In [ ]:
# ==============================================================================
# MODELADO GEOMÉTRICO 3D DEL DATA CENTER (ANSI/TIA-942 & ANSI/TIA-568)
# Autor: Gerson Misael Pintado Huaman | Docente: Ing. Javier Eduardo Jaramillo
# ==============================================================================

def create_datacenter_3d_simulation():
    fig = go.Figure()

    # 1. PISO TÉCNICO ELEVADO (Raised Floor)
    # Dimensiones de la sala: 8m x 6m
    x_floor = [-4, 4, 4, -4, -4]
    y_floor = [-3, -3, 3, 3, -3]
    z_floor = [0, 0, 0, 0, 0]
    
    fig.add_trace(go.Scatter3d(
        x=x_floor, y=y_floor, z=z_floor,
        mode='lines',
        line=dict(color='#38bdf8', width=5),
        name='Piso Técnico (Perímetro Sala)',
        hoverinfo='text',
        text='Sala Blanca - Data Center (8m x 6m x 3.5m)'
    ))

    # Cuadrícula de baldosas del piso técnico
    for x in np.linspace(-4, 4, 9):
        fig.add_trace(go.Scatter3d(
            x=[x, x], y=[-3, 3], z=[0, 0],
            mode='lines',
            line=dict(color='#1e293b', width=1.5),
            showlegend=False,
            hoverinfo='none'
        ))
    for y in np.linspace(-3, 3, 7):
        fig.add_trace(go.Scatter3d(
            x=[-4, 4], y=[y, y], z=[0, 0],
            mode='lines',
            line=dict(color='#1e293b', width=1.5),
            showlegend=False,
            hoverinfo='none'
        ))

    # Función auxiliar para dibujar prismas (racks)
    def add_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, name, color, opacity=0.85, info=''):
        # 8 vértices
        x = [x_min, x_max, x_max, x_min, x_min, x_max, x_max, x_min]
        y = [y_min, y_min, y_max, y_max, y_min, y_min, y_max, y_max]
        z = [z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max]
        
        # 12 triángulos de las 6 caras
        i = [7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2]
        j = [3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3]
        k = [0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6]
        
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z, i=i, j=j, k=k,
            color=color,
            opacity=opacity,
            name=name,
            hoverinfo='text',
            text=f"<b>{name}</b><br>{info}"
        ))

    # 2. RACKS DE SERVIDORES Y STORAGE (EDA - 42U)
    racks = [
        ("Rack 01 - DB Clúster (PostgreSQL/Oracle)", -2.4, -1.6, -1.8, -1.0, 0, 2.1, '#0f172a', 'EDA: Servidores Base de Datos | ToR Leaf-01'),
        ("Rack 02 - App Containers (Docker/K8s)", -0.8, 0.0, -1.8, -1.0, 0, 2.1, '#0f172a', 'EDA: Virtualización y Microservicios | ToR Leaf-02'),
        ("Rack 03 - Storage SAN All-Flash NVMe", 0.8, 1.6, -1.8, -1.0, 0, 2.1, '#0f172a', 'EDA: Cabina SAN 100TB | ToR Leaf-03'),
        ("Rack 04 - Backup & Servicios Aduana", 2.4, 3.2, -1.8, -1.0, 0, 2.1, '#0f172a', 'EDA: Backup LTO + Trámites SUNAT | ToR Leaf-04'),
        ("Rack MDA - Switches Spine Centrales", -0.5, 0.5, 1.4, 2.2, 0, 2.1, '#1e3a8a', 'MDA: 2x Switches Spine 100G + Core Enrutamiento')
    ]

    for name, x0, x1, y0, y1, z0, z1, col, inf in racks:
        add_box(fig, x0, x1, y0, y1, z0, z1, name, col, opacity=0.9, info=inf)

    # 3. SWITCHES TOP OF RACK (ToR Leaf) y SWITCHES SPINE (Core)
    tor_positions = [
        ("Switch Leaf 01 (ToR)", -2.0, -1.4, 2.05, '#06b6d4', '48x10G RJ45 + 6x100G QSFP28 Uplink'),
        ("Switch Leaf 02 (ToR)", -0.4, -1.4, 2.05, '#06b6d4', '48x10G RJ45 + 6x100G QSFP28 Uplink'),
        ("Switch Leaf 03 (ToR)", 1.2, -1.4, 2.05, '#06b6d4', '48x10G RJ45 + 6x100G QSFP28 Uplink'),
        ("Switch Leaf 04 (ToR)", 2.8, -1.4, 2.05, '#06b6d4', '48x10G RJ45 + 6x100G QSFP28 Uplink')
    ]
    spine_positions = [
        ("Switch Spine 01 (Core MDA)", -0.3, 1.8, 1.9, '#f59e0b', '32x100G QSFP28 Spine Switch - Latencia 400ns'),
        ("Switch Spine 02 (Core MDA)", 0.3, 1.8, 1.9, '#f59e0b', '32x100G QSFP28 Spine Switch - Redundante Activo/Activo')
    ]

    for name, x, y, z, col, inf in tor_positions:
        fig.add_trace(go.Scatter3d(
            x=[x], y=[y], z=[z],
            mode='markers+text',
            marker=dict(size=8, color=col, symbol='diamond'),
            name=name,
            hoverinfo='text',
            text=f"<b>{name}</b><br>{inf}",
            showlegend=True
        ))

    for name, x, y, z, col, inf in spine_positions:
        fig.add_trace(go.Scatter3d(
            x=[x], y=[y], z=[z],
            mode='markers+text',
            marker=dict(size=10, color=col, symbol='square'),
            name=name,
            hoverinfo='text',
            text=f"<b>{name}</b><br>{inf}",
            showlegend=True
        ))

    # 4. BANDEJAS PORTACABLES AÉREAS (CABLE TRAYS A 2.6m DE ALTURA)
    tray_x = [-3.0, 3.5, 3.5, -3.0, -3.0]
    tray_y = [-1.4, -1.4, 1.8, 1.8, -1.4]
    tray_z = [2.6, 2.6, 2.6, 2.6, 2.6]
    fig.add_trace(go.Scatter3d(
        x=tray_x, y=tray_y, z=tray_z,
        mode='lines',
        line=dict(color='#e2e8f0', width=8),
        name='Bandeja Aérea Portacables (Tipo Canastilla)',
        hoverinfo='text',
        text='Bandeja de Malla de Acero Electrozincado (Suspendida a 2.60m)'
    ))

    # 5. BACKBONE FIBRA ÓPTICA OM4 (100G SPINE-LEAF MESH)
    for l_name, lx, ly, lz, _, _ in tor_positions:
        for s_name, sx, sy, sz, _, _ in spine_positions:
            # Cable sube a bandeja (z=2.6), cruza y baja
            cx = [lx, lx, sx, sx]
            cy = [ly, ly, sy, sy]
            cz = [lz, 2.6, 2.6, sz]
            fig.add_trace(go.Scatter3d(
                x=cx, y=cy, z=cz,
                mode='lines',
                line=dict(color='#00f0ff', width=4),
                name='Backbone Fibra Óptica OM4 (100G MPO)',
                hoverinfo='text',
                text=f"Enlace MPO 100GbE: {l_name} ➔ {s_name}<br>Medio: OM4 Multimodo 12 fibras (850nm)<br>Atenuación: < 0.2 dB | MTU: 9000",
                showlegend=(l_name == tor_positions[0][0] and s_name == spine_positions[0][0])
            ))

    # 6. ENLACE WAN EN FIBRA MONOMODO OS2 (Acometida a Cuarto de Entrada ER)
    er_x, er_y, er_z = -3.8, 2.5, 2.6
    fig.add_trace(go.Scatter3d(
        x=[-0.3, -3.8], y=[1.8, 2.5], z=[2.6, 2.6],
        mode='lines',
        line=dict(color='#eab308', width=6),
        name='Acometida WAN Fibra OS2 Monomodo',
        hoverinfo='text',
        text='Enlace Externo OS2 (9/125 µm): MDA ➔ ER (Operadores Internet / APN Paita)<br>Velocidad: 10G/40G WAN | Latencia: 0.1ms'
    ))
    fig.add_trace(go.Scatter3d(
        x=[er_x], y=[er_y], z=[er_z],
        mode='markers+text',
        marker=dict(size=10, color='#eab308', symbol='cross'),
        name='Cuarto de Entrada (ER)',
        hoverinfo='text',
        text='ER: Punto de Demarcación de Operadores (ISPs Duales)',
        showlegend=True
    ))

    # 7. CABLEADO DE COBRE CATEGORÍA 6A (Parcheo de Servidores a ToR)
    for l_name, lx, ly, lz, _, _ in tor_positions:
        # Patch cords de servidores en el mismo rack (de z=0.4 a 1.8 hacia switch ToR en 2.05)
        for s_z in [0.5, 0.9, 1.3, 1.7]:
            fig.add_trace(go.Scatter3d(
                x=[lx, lx + 0.15, lx],
                y=[ly, ly, ly],
                z=[s_z, (s_z + 2.05)/2, 2.05],
                mode='lines',
                line=dict(color='#f97316', width=3),
                name='Parcheo Cobre Cat 6A STP (10G ToR)',
                hoverinfo='text',
                text=f"Patch Cord Cat 6A Blindado (F/UTP): Servidor U-Slot ➔ {l_name}<br>Velocidad: 10GBASE-T (10 Gbps)<br>Longitud: 1.8 m | Conector: RJ45 apantallado",
                showlegend=(l_name == tor_positions[0][0] and s_z == 0.5)
            ))

    # 8. CONFINAMIENTO DE PASILLO FRÍO (Cold Aisle Containment)
    # Cerramiento frontal de los racks con paredes transparentes
    add_box(fig, -2.6, 3.4, -0.9, -0.1, 0, 2.3, 
            "Confinamiento Pasillo Frío", '#38bdf8', opacity=0.25, 
            info="Pasillo Frío Confinado (Cold Aisle)<br>Temp: 20°C - 22°C | Flujo inyectado desde piso falso")

    # Flechas / Partículas de Flujo Frío (Azul) y Calor (Rojo)
    for fx in [-1.5, 0.5, 2.0]:
        fig.add_trace(go.Scatter3d(
            x=[fx, fx], y=[-0.5, -0.5], z=[0.1, 1.8],
            mode='lines+markers',
            marker=dict(size=4, color='#38bdf8'),
            line=dict(color='#38bdf8', width=4, dash='dot'),
            name='Flujo de Aire Frío (20°C)',
            hoverinfo='text',
            text='Aire Frío impulsado por CRAC bajo piso técnico',
            showlegend=(fx == -1.5)
        ))
        # Aire caliente saliendo por detrás (y = -2.1)
        fig.add_trace(go.Scatter3d(
            x=[fx, fx], y=[-2.1, -2.1], z=[1.5, 2.9],
            mode='lines+markers',
            marker=dict(size=4, color='#ef4444'),
            line=dict(color='#ef4444', width=4, dash='dash'),
            name='Flujo de Escape Caliente (35°C)',
            hoverinfo='text',
            text='Aire Caliente expulsado hacia retorno de CRAC',
            showlegend=(fx == -1.5)
        ))

    # 9. CONFIGURACIÓN DE LA ESCENA Y CÁMARA 3D
    fig.update_layout(
        title=dict(
            text="<b>SIMULACIÓN 3D: SISTEMA DE TRANSPORTE DE INFORMACIÓN EN DATA CENTER</b><br>"
                 "<sup>IESTP Hermanos Cárcamo – Paita | Autor: Gerson Misael Pintado Huaman | Docente: Ing. Javier Jaramillo</sup>",
            font=dict(size=16, color='#f8fafc'),
            x=0.5, y=0.96
        ),
        paper_bgcolor='#0b132b',
        scene=dict(
            bgcolor='#0b132b',
            xaxis=dict(title='Ancho Sala (m)', backgroundcolor='#0f172a', gridcolor='#334155', showbackground=True, zerolinecolor='#64748b'),
            yaxis=dict(title='Largo Sala (m)', backgroundcolor='#0f172a', gridcolor='#334155', showbackground=True, zerolinecolor='#64748b'),
            zaxis=dict(title='Altura (m)', backgroundcolor='#0f172a', gridcolor='#334155', showbackground=True, zerolinecolor='#64748b'),
            camera=dict(
                eye=dict(x=-1.6, y=-1.8, z=1.3),
                center=dict(x=0, y=0, z=0.2)
            ),
            aspectmode='manual',
            aspectratio=dict(x=1.3, y=1.0, z=0.6)
        ),
        legend=dict(
            font=dict(color='#e2e8f0', size=10),
            bgcolor='rgba(15, 23, 42, 0.8)',
            bordercolor='#38bdf8',
            borderwidth=1,
            x=0.02, y=0.95
        ),
        margin=dict(l=0, r=0, b=0, t=60)
    )

    return fig

fig_sim = create_datacenter_3d_simulation()
# Mostrar figura en Colab
fig_sim.show()

# Exportar como archivo HTML interactivo autónomo
html_path = "Simulacion_3D_DataCenter_Pintado.html"
fig_sim.write_html(html_path)
print(f"✅ Archivo de simulación interactiva 3D generado con éxito: {html_path}")


In [ ]:
# ==============================================================================
# VISOR 3D WEBGL INTERACTIVO EN TIEMPO REAL (THREE.JS ENGINE)
# ==============================================================================
# Esta celda permite ejecutar la simulación tridimensional fotorrealista directamente
# en Google Colab con luces dinámicas, pasillo frío confinado y paquetes ópticos:

from IPython.display import HTML, display

colab_viewer_code = '''
<div style="text-align: center; margin-bottom: 8px;">
  <span style="font-family: monospace; font-size: 12px; color: #00f0ff; background: #0b132b; padding: 4px 12px; border-radius: 8px; border: 1px solid #0284c7;">
    ⚡ SIMULADOR 3D WEBGL ACTIVO: ROTAR 360° | SCROLL ZOOM | CLIC EN COMPONENTES
  </span>
</div>
<iframe style="width: 100%; height: 650px; border-radius: 16px; border: 2px solid #00f0ff; box-shadow: 0 0 24px rgba(0, 240, 255, 0.35);"
        src="https://htmlpreview.github.io/?https://raw.githubusercontent.com/"
        srcdoc='''' + open('Simulacion_3D_Interactiva_Web.html', 'r', encoding='utf-8').read().replace("'", "&#39;") + '''''>
</iframe>
'''

try:
    with open('Simulacion_3D_Interactiva_Web.html', 'r', encoding='utf-8') as f:
        html_data = f.read()
    print("✅ Motor 3D WebGL listo para ejecutar en Google Colab.")
    display(HTML(f'<div style="background: #060b18; padding: 12px; border-radius: 16px; border: 2px solid #00f0ff;"><iframe style="width: 100%; height: 650px; border: none; border-radius: 12px;" srcdoc="{html_data.replace(chr(34), "&quot;")}"></iframe></div>'))
except Exception as e:
    print("💡 Para visualizar en cualquier navegador local, abre el archivo Simulacion_3D_Interactiva_Web.html con doble clic.")


---
## 📊 MEMORIA DE CÁLCULO DE TELECOMUNICACIONES
### 1. Presupuesto Óptico de Atenuación (Optical Power Budget)
Conforme a la norma **ANSI/TIA-568.3-D**, la atenuación total máxima permitida para un enlace de fibra multimodo OM4 a 850 nm se calcula como:

$$A_{\text{total}} = (\alpha \cdot L) + (N_c \cdot A_c) + (N_s \cdot A_s)$$

Donde:
* $\alpha$: Coeficiente de atenuación de la fibra OM4 a 850 nm = $3.0\text{ dB/km} = 0.003\text{ dB/m}$
* $L$: Longitud del enlace = $15\text{ m}$ (inter-rack Spine-Leaf)
* $N_c$: Número de pares de conectores MPO = $2$
* $A_c$: Pérdida por conector MPO de baja pérdida = $0.35\text{ dB}$
* $N_s$: Número de empalmes por fusión = $0$ (troncales preconectorizados de fábrica)
* $A_s$: Pérdida por empalme = $0.05\text{ dB}$


In [ ]:
# Cálculo computacional de atenuación y ancho de banda
alpha_om4 = 0.003 # dB/m a 850nm
L_link = 15.0     # metros
N_c = 2           # 2 conectores MPO
A_c = 0.35        # dB por conector MPO
N_s = 0           # enlaces preconectorizados

attenuation_om4 = (alpha_om4 * L_link) + (N_c * A_c)
margin_ieee_100g = 1.9 # Presupuesto máximo IEEE 802.3ba 100GBASE-SR4 (dB)
headroom = margin_ieee_100g - attenuation_om4

print(f"--- ANÁLISIS DE ENLACE FIBRA ÓPTICA OM4 (100G SPINE-LEAF) ---")
print(f"Atenuación total del enlace: {attenuation_om4:.4f} dB")
print(f"Límite máximo IEEE 802.3ba: {margin_ieee_100g:.2f} dB")
print(f"Margen de seguridad (Headroom): {headroom:.4f} dB (ÓPTIMO)")
if headroom > 0:
    print("✅ ESTADO: Enlace cumple holgadamente los estándares de certificación.")


---
## ⚡ COMPARATIVA DE LATENCIA: SPINE-LEAF VS. JERÁRQUICA TRADICIONAL
| Parámetro de Red | Arquitectura Spine-Leaf (Propuesta) | Arquitectura Jerárquica 3 Capas | Beneficio Obtenido |
| :--- | :--- | :--- | :--- |
| **Saltos de Conmutación (Hops)** | **Constante: 2 saltos** entre cualquier servidor | Variable (de 2 a 6 saltos) | **Latencia predecible y consistente** |
| **Latencia Promedio Este-Oeste** | **< 1.0 µs (microsegundo)** | 15.0 a 35.0 µs | **Reducción del 95% en tiempos de espera** |
| **Protocolo de Prevención Bucles**| **ECMP (Equal Cost Multi-Path)** L3 | STP (Spanning Tree Protocol) | **Cero enlaces bloqueados (100% throughput)**|
| **Escalabilidad de Racks** | **Horizontal (añadir Leaves)** sin parar red | Vertical (reemplazo costoso de Core) | **Alta escalabilidad sin downtime** |
| **Medio Backbone Recomendado** | **Fibra OM4 MPO 100G / OS2** | Cobre / Fibra OM3 10G | **Capacidad de actualización a 400G** |

---
## 🏆 CONCLUSIONES DEL ESTUDIANTE
1. **Modelado en 3D:** La simulación gráfica tridimensional permite visualizar con exactitud la separación de corrientes de aire (pasillo frío confinado a 20°C vs pasillo caliente de escape) y la distribución ordenada de los medios físicos en bandejas de malla aéreas a 2.60 m.
2. **Cobre vs. Fibra:** Queda validado el uso de **Cobre Cat 6A blindado** para tramos cortos de 1.8m (servidores a switch Top of Rack), lo que reduce costos en transceptores, mientras que la **Fibra Óptica OM4/OS2** es indispensable en el backbone para garantizar 100 Gbps inmunes al ruido electromagnético de la zona portuaria de Paita.
